# tags

> Reading thinking and tool calls out of plain text.

Not every transport can carry tool schemas on the wire. Some models are trained to emit a call as `<tool_call>{json}</tool_call>` in the reply body instead, and some emit thinking as `<think>...</think>`. This module puts the schemas into the system prompt, reads the calls back out of the text, and does the same job incrementally for a stream.

The batch parser and the streaming splitter must agree, so they share their tag tables.

In [ ]:
#| default_exp tags

In [ ]:
#| export
import json, re, uuid
from fastcore.all import L

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail

## Thinking

`<think>` blocks come out first, before anything looks for a call. A block left unterminated means the reply hit its token cap mid-thought; the text after the opening tag is still the thinking, so it is kept rather than dropped.

In [ ]:
#| export
_think_re = re.compile(r'<think>(.*?)</think>', re.DOTALL)

def split_think(text):
    "Split `<think>...</think>` out of `text`, returning `(clean_text, thought)`."
    text = text or ''
    ths = [m.strip() for m in _think_re.findall(text)]
    text = _think_re.sub('', text)
    if '<think>' in text:            # unterminated, e.g. cut off at the token cap
        text, _, rest = text.partition('<think>')
        ths.append(rest.strip())
    return text.strip('\n'), '\n'.join(th for th in ths if th)

In [ ]:
test_eq(split_think('<think>weighing it</think>the answer'), ('the answer', 'weighing it'))
test_eq(split_think('no tags here'), ('no tags here', ''))
test_eq(split_think('<think>one</think>mid<think>two</think>'), ('mid', 'one\ntwo'))
test_eq(split_think('answer<think>cut off here'), ('answer', 'cut off here'))
test_eq(split_think(None), ('', ''))

## A call in the text

Models disagree about what the argument key is called, so all three spellings are accepted, and a JSON string where an object was expected is decoded rather than rejected.

In [ ]:
#| export
TAG_ARG_KEYS = ('arguments', 'input', 'parameters')

def tag_args(d):
    "The argument dict of a tagged call, decoding the JSON string some models send instead."
    for k in TAG_ARG_KEYS:
        if (v := d.get(k)) is None: continue
        if isinstance(v, str):
            try: v = json.loads(v)
            except json.JSONDecodeError: continue
        if isinstance(v, dict): return v
    return {}

def mk_tag_tc(s):
    "A tool_call dict from the JSON inside a `<tool_call>` block, or None."
    try: d = json.loads(s)
    except (json.JSONDecodeError, TypeError): return None
    if not isinstance(d, dict) or not d.get('name'): return None
    return {'id': f'call_{uuid.uuid4().hex[:8]}', 'type': 'function',
            'function': {'name': d['name'], 'arguments': tag_args(d)}}

In [ ]:
test_eq(tag_args({'arguments': {'a': 1}}), {'a': 1})
test_eq(tag_args({'input': '{"a": 1}'}), {'a': 1})     # a JSON string, decoded
test_eq(tag_args({'parameters': {'b': 2}}), {'b': 2})
test_eq(tag_args({'arguments': 'not json'}), {})
test_eq(tag_args({}), {})

In [ ]:
tc = mk_tag_tc('{"name": "add", "arguments": {"a": 1}}')
test_eq(tc['function'], {'name': 'add', 'arguments': {'a': 1}})
assert tc['id'].startswith('call_')
test_eq(mk_tag_tc('{"arguments": {}}'), None)   # no name is not a call
test_eq(mk_tag_tc('not json'), None)
test_eq(mk_tag_tc('[1, 2]'), None)

Some models drop the tags and emit the bare object. That is worth reading as a call, but only under a narrow rule: the object has to be the *entire* reply and carry both a name and an argument key. A reply that describes JSON must not be executed.

In [ ]:
#| export
_res_tags = ('tool_result', 'tool_results', 'function_result', 'function_results',
             'tool_function_result', 'tool_function_results',
             'tool_function_call', 'tool_function_calls',
             'function_call', 'function_calls', 'tool_use')
_toolres_re = re.compile('|'.join(rf'</?{t}>>?' for t in _res_tags) + r'|</?tool_call>', re.I)
_toolcall_re = re.compile(r'<tool_call>\s*(.*?)\s*</tool_call>', re.DOTALL)
_fence_re = re.compile(r'^```(?:json)?\s*|\s*```$')

def lone_tag_tc(text):
    "A whole reply that is one bare call object: what the tags asked for, without the tags."
    s = _fence_re.sub('', (text or '').strip()).strip()
    if not (s.startswith('{') and s.endswith('}')): return None
    try: d = json.loads(s)
    except json.JSONDecodeError: return None
    if not isinstance(d, dict) or not d.get('name'): return None
    if not any(k in d for k in TAG_ARG_KEYS): return None
    return mk_tag_tc(s)

In [ ]:
test_eq(lone_tag_tc('{"name": "add", "arguments": {"a": 1}}')['function']['name'], 'add')
test_eq(lone_tag_tc('```json\n{"name": "add", "input": {}}\n```')['function']['name'], 'add')
test_eq(lone_tag_tc('Here is the call: {"name": "add", "arguments": {}}'), None)  # not the whole reply
test_eq(lone_tag_tc('{"name": "add"}'), None)                                    # no argument key
test_eq(lone_tag_tc('{"a": 1}'), None)
test_eq(lone_tag_tc(''), None)

`parse_tool_tags` is the batch entry point: tagged calls first, the bare-object fallback only when there were none. It also strips result markup a model may have invented, so an imagined result stays ordinary prose instead of being read back as a tool message.

In [ ]:
#| export
def parse_tool_tags(text):
    "Parse Hermes and Qwen style `<tool_call>{json}</tool_call>` blocks, returning `(clean_text, tool_calls)`."
    tcs = [tc for m in _toolcall_re.findall(text or '') if (tc := mk_tag_tc(m))]
    if not tcs and (tc := lone_tag_tc(text)): return '', [tc]
    return _toolres_re.sub('', _toolcall_re.sub('', text or '')).strip('\n'), tcs

In [ ]:
txt, tcs = parse_tool_tags('Let me look.\n<tool_call>{"name": "ls", "arguments": {}}</tool_call>')
test_eq(txt, 'Let me look.')
test_eq([t['function']['name'] for t in tcs], ['ls'])

In [ ]:
test_eq(parse_tool_tags('just prose'), ('just prose', []))
txt, tcs = parse_tool_tags('<tool_call>{"name":"a","arguments":{}}</tool_call>'
                           '<tool_call>{"name":"b","arguments":{}}</tool_call>')
test_eq([t['function']['name'] for t in tcs], ['a', 'b'])

In [ ]:
# invented result tags go; the text stays prose, and no tool call is claimed
test_eq(parse_tool_tags('done <tool_result>42</tool_result>'), ('done 42', []))
# the bare-object fallback fires only when no tagged call was found
test_eq(parse_tool_tags('{"name": "add", "arguments": {"a": 1}}')[1][0]['function']['name'], 'add')

When a model narrates a call instead of emitting one, the parser takes nothing and the caller is left with prose where it expected an action. `tag_call_shape` spots that so the caller can say which model is unreliable on this channel, rather than silently doing nothing.

In [ ]:
#| export
def tag_call_shape(text, names=None):
    "Does `text` still show a call the parser did not take? `names` limits it to tools that exist."
    t = text or ''
    if '<tool_call' in t: return True
    if not any(f'"{k}"' in t for k in TAG_ARG_KEYS): return False
    pat = '|'.join(re.escape(str(n)) for n in (names or []) if n) if names is not None else r'[A-Za-z_]\w*'
    return bool(pat) and bool(re.search(rf'"name"\s*:\s*"(?:{pat})"', t))

In [ ]:
test_eq(tag_call_shape('I would call <tool_call> here'), True)
test_eq(tag_call_shape('maybe {"name": "add", "arguments": {}}', ['add']), True)
test_eq(tag_call_shape('maybe {"name": "add", "arguments": {}}', ['other']), False)  # no such tool
test_eq(tag_call_shape('plain prose'), False)
test_eq(tag_call_shape('{"name": "add"}'), False)   # no argument key, so not call-shaped

## Putting the schemas in the prompt

The prompt is blunt about stopping after the call, because the failure it prevents is expensive: a model that guesses the result and keeps writing produces a plausible answer built on a number nobody computed.

In [ ]:
#| export
TAG_TOOLS_SP = """

# Tools

You can call the functions below. Their signatures are given as JSON schemas inside \
<tools></tools>:

<tools>
{tools}
</tools>

To call one, emit a JSON object with the function's name and its arguments inside \
<tool_call></tool_call>. Then end your reply immediately and wait:

<tool_call>
{{"name": "the_function_name", "arguments": {{"first": "value"}}}}
</tool_call>

Nothing may follow </tool_call> in the same message: not a comment, not a guess at what the \
result will be, not another call. Stop there. Call one function at a time.

Do not describe the call in prose as well as emitting it, and never invent a result -- the real \
one comes back in the next message, under a "## Tool result (name)" heading. That heading is the \
only form a result ever takes: do not write result markup of your own, do not wrap a result in \
tags, and do not copy a result back into your reply. Say what you concluded from it, not what \
it said."""

def tag_tools_sp(toolspecs, sp='', template=TAG_TOOLS_SP):
    "`sp` plus the tag protocol and `toolspecs` as JSON, for a transport that can't carry tools."
    if not toolspecs: return sp
    block = '\n'.join(json.dumps(t, ensure_ascii=False) for t in toolspecs)
    return (sp or '') + template.format(tools=block)

In [ ]:
spec = {'type': 'function', 'function': {'name': 'add', 'parameters': {}}}
out = tag_tools_sp([spec], 'Be brief.')
assert out.startswith('Be brief.') and '"name": "add"' in out and '<tool_call>' in out
test_eq(tag_tools_sp([], 'Be brief.'), 'Be brief.')   # no tools, no protocol

## Streaming

A stream arrives in arbitrary slices, so a tag can be cut in half between two deltas. `StreamSplit` holds back the longest suffix of its buffer that could still grow into a tag, emits everything before it, and resumes on the next delta. Text inside a `<tool_call>` block is withheld entirely — the caller sees a parsed call, never the raw JSON.

In [ ]:
#| export
_tags = ('<think>', '</think>', '<tool_call>', '</tool_call>',
         *(f'<{n}>' for n in _res_tags), *(f'</{n}>' for n in _res_tags))

class StreamSplit:
    "Stateful splitter: `<think>` becomes thought chunks, `<tool_call>` blocks are held back and parsed."
    def __init__(self):
        self.buf, self.state, self.text, self.thought = '', 'text', '', ''
        self.tool_calls, self._tc_buf, self._strip = [], '', False

    def _held(self):
        "Length of the longest `buf` suffix that could still become a tag."
        for n in range(min(len(self.buf), max(map(len, _tags)) - 1), 0, -1):
            if any(t.startswith(self.buf[-n:]) for t in _tags): return n
        return 0

    def _emit_text(self, out):
        out = _toolres_re.sub('', out)       # invented result markup: the streamed path never
        if self._strip: out = out.lstrip('\n')   # reaches `parse_tool_tags`, so it is dropped here
        if not out: return None
        self._strip = False; self.text += out
        return {'content': [{'type': 'text', 'text': out}]}

    def feed(self, s):
        "Consume a text delta and yield chunk dicts."
        self.buf += s
        while True:
            if self.state == 'text':
                cands = [(k, t, st) for k, t, st in ((self.buf.find('<think>'), '<think>', 'think'),
                                                     (self.buf.find('<tool_call>'), '<tool_call>', 'tool')) if k >= 0]
                if not cands:
                    n = self._held()
                    out, self.buf = self.buf[:len(self.buf) - n], self.buf[len(self.buf) - n:]
                    if (c := self._emit_text(out)): yield c
                    return
                k, tag, st = min(cands)
                out, self.buf, self.state = self.buf[:k], self.buf[k + len(tag):], st
                if (c := self._emit_text(out)): yield c
            elif self.state == 'think':
                k = self.buf.find('</think>')
                if k < 0:
                    n = self._held()
                    out, self.buf = self.buf[:len(self.buf) - n], self.buf[len(self.buf) - n:]
                    if out: self.thought += out; yield {'channels': {'thought': out}}
                    return
                out, self.buf, self.state, self._strip = self.buf[:k], self.buf[k + len('</think>'):], 'text', True
                if out: self.thought += out; yield {'channels': {'thought': out}}
            else:                                    # inside a tool call
                k = self.buf.find('</tool_call>')
                if k < 0:
                    n = self._held()
                    self._tc_buf += self.buf[:len(self.buf) - n]; self.buf = self.buf[len(self.buf) - n:]
                    return
                self._tc_buf += self.buf[:k]
                self.buf, self.state, self._strip = self.buf[k + len('</tool_call>'):], 'text', True
                if (tc := mk_tag_tc(self._tc_buf)): self.tool_calls.append(tc)
                self._tc_buf = ''

    def finish(self):
        "Flush leftovers: unterminated think becomes thought, an unclosed tool block is parsed if it can be."
        s, self.buf = self.buf, ''
        if self.state == 'think':
            if s: self.thought += s; yield {'channels': {'thought': s}}
        elif self.state == 'tool':
            if (tc := mk_tag_tc(self._tc_buf + s)): self.tool_calls.append(tc)
            self._tc_buf = ''
        elif (c := self._emit_text(s)): yield c

In [ ]:
def _run(deltas):
    "Feed `deltas` through a fresh splitter and return it with the chunks it produced."
    s = StreamSplit()
    chunks = [c for d in deltas for c in s.feed(d)] + list(s.finish())
    return s, chunks

s, _ = _run(['<think>', 'wei', 'ghing it', '</think>', 'the ', 'answer'])
test_eq((s.thought, s.text), ('weighing it', 'the answer'))

In [ ]:
# a tag split across three deltas is still one tag
s, _ = _run(['<thi', 'nk>deep</th', 'ink>done'])
test_eq((s.thought, s.text), ('deep', 'done'))

In [ ]:
s, _ = _run(['Looking.<tool_call>{"name": "ls", ', '"arguments": {}}</tool_call>'])
test_eq(s.text, 'Looking.')                       # the JSON never reaches the caller as text
test_eq([t['function']['name'] for t in s.tool_calls], ['ls'])

In [ ]:
s, _ = _run(['<think>cut off mid-thought'])       # unterminated: flushed by `finish`
test_eq((s.thought, s.text), ('cut off mid-thought', ''))
s, _ = _run(['<tool_call>{"name": "ls", "arguments": {}}'])   # unclosed, but parseable
test_eq([t['function']['name'] for t in s.tool_calls], ['ls'])

In [ ]:
_, chunks = _run(['<think>why</think>because'])
test_eq(chunks, [{'channels': {'thought': 'why'}},
                 {'content': [{'type': 'text', 'text': 'because'}]}])
test_eq(_run(['plain text'])[0].text, 'plain text')

Native tool calls stream as fragments too: OpenAI sends a name and its arguments across many deltas, keyed by index. `acc_tc` folds them back into whole calls.

In [ ]:
#| export
def acc_tc(acc, deltas):
    "Fold streamed OpenAI `tool_calls` deltas into `acc`, a list of partial tool_call dicts."
    for d in deltas or []:
        i = d.get('index', 0)
        while len(acc) <= i: acc.append({'id': None, 'type': 'function',
                                         'function': {'name': '', 'arguments': ''}})
        if d.get('id'): acc[i]['id'] = d['id']
        f = d.get('function') or {}
        if f.get('name'): acc[i]['function']['name'] += f['name']
        if f.get('arguments'): acc[i]['function']['arguments'] += f['arguments']

In [ ]:
acc = []
acc_tc(acc, [{'index': 0, 'id': 'c1', 'function': {'name': 'ad'}}])
acc_tc(acc, [{'index': 0, 'function': {'name': 'd', 'arguments': '{"a":'}}])
acc_tc(acc, [{'index': 0, 'function': {'arguments': ' 1}'}}])
test_eq(acc, [{'id': 'c1', 'type': 'function', 'function': {'name': 'add', 'arguments': '{"a": 1}'}}])

In [ ]:
acc = []
acc_tc(acc, [{'index': 1, 'id': 'c2', 'function': {'name': 'b'}}])   # index 1 arrives first
test_eq(len(acc), 2)
test_eq(acc[0]['function']['name'], '')       # index 0 is a placeholder until it arrives
test_eq(acc[1]['function']['name'], 'b')
acc_tc(acc, None)
test_eq(len(acc), 2)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()